In [ ]:
"""
TANSALTION PROJECT - PREPROCESSING
1. Loading dataset
2. Text moderation
3. Language detection
4. Preprocessing (cleaning, lowercasing, emoji & punctuation handling)


🐼 pandas → handles your data
🔍 re → cleans unwanted text
😀 emoji → removes emojis
🌐 langdetect → finds which language
🧾 loguru → tracks what’s happening
✂️ nltk → helps process text (split, clean, remove stopwords)

💬 What is punkt?
🧠 punkt is a tokenizer model in the NLTK library.
That means it helps split a sentence into words or sentences.

What are Stopwords?
🧠 Stopwords are common words that don’t add real meaning to a sentence.
Example: “is”, “am”, “are”, “the”, “and”, “to”, etc.
They are often removed to keep only the important words for translation or analysis.


"""

In [12]:
# =========================================
# PREPROCESSING
# =========================================

import os
import re
import emoji
import pandas as pd
from langdetect import detect
from loguru import logger
import nltk

In [2]:
# Download resources (run only once)
#nltk.download('punkt') -- this one is compulsory
##nltk.download('stopwords') #Real chat messages (for display or reply) - no need to remove stopwords

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

In [18]:
# -----------------------------------------
# 1️.Safe Folder + File Setup
# -----------------------------------------

# ------------------- PATH SETUP -------------------
BASE_DIR = r"E:\HOPE\AI Course Tamil\translation_project"
DATA_DIR = os.path.join(BASE_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)

DATA_FILE = os.path.join(DATA_DIR, "translation_dataset.csv")
OUTPUT_FILE = os.path.join(DATA_DIR, "preprocessed_dataset.csv")

# ------------------- LOAD DATA -------------------
if not os.path.exists(DATA_FILE):
    logger.error(f"Dataset file not found at: {DATA_FILE}. Please add your dataset and re-run.")
else:
    df = pd.read_csv(DATA_FILE, encoding="utf-8")
    logger.info(f"Loaded dataset successfully from: {DATA_FILE}")

2025-11-03 15:05:16.729 | INFO     | __main__:<module>:18 - Loaded dataset successfully from: E:\HOPE\AI Course Tamil\translation_project\data\translation_dataset.csv


In [19]:
df.head(5)

,id,input_text,target_language,human_translated_text
0,1,அழுமூஞ்சியாக இருக்காதே.,en,Don’t be a cry-baby. / Don’t be a fusspot.
1,2,சிணுங்குவதை நிறுத்து.,en,Stop whining. / Stop whimpering.
2,3,நீ மிகவும் குறும்புக்காரி.,en,You are very naughty.
3,4,இந்த பையன் மிகவும் தேட்தடச் சேயவன்.,en,This boy is very mischievous; troublesome.
4,5,ஒழுங்கா இரு.,en,Behave yourself. / Keep quiet.


In [21]:
#-----------------------------------------
# 2️ . Clean Text Function
# -----------------------------------------
def clean_text(text):
    if not isinstance(text, str):
        return ""
    try:
        # Remove URLs, mentions
        text = re.sub(r"http\S+|www\S+|@\S+", "", text)
        # Remove emojis safely
        text = emoji.replace_emoji(text, replace='')
        # Remove unwanted chars (supports Tamil + Hindi Unicode)
        text = re.sub(r"[^a-zA-Z0-9\u0B80-\u0BFF\u0900-\u097F\s.,!?]", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text
    except Exception as e:
        logger.error(f"Error cleaning text: {e}")
        return ""


In [22]:
# -----------------------------------------
# 3️. Detect Language (safe)
# -----------------------------------------
def detect_language(text):
    try:
        return detect(text)
    except Exception:
        return "unknown"

In [23]:
# -----------------------------------------
# 4️. Moderation Function
# -----------------------------------------
def moderate_text(text):
    banned_words = ["hate", "kill", "terror", "violence"]
    try:
        for word in banned_words:
            if word.lower() in text.lower():
                return "rejected"
        return "approved"
    except Exception:
        return "unknown"


In [24]:
# -----------------------------------------
# 5️. Apply Preprocessing Pipeline
# -----------------------------------------
if not df.empty:
    df["clean_text"] = df["input_text"].apply(clean_text)
    df["detected_lang"] = df["clean_text"].apply(detect_language)
    df["moderation_status"] = df["clean_text"].apply(moderate_text)

    # Save safely
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
    logger.info(f"Preprocessed dataset saved successfully: {OUTPUT_FILE}")
else:
    logger.warning("Dataset is empty. Please add data and rerun.")

2025-11-03 15:06:23.627 | INFO     | __main__:<module>:11 - Preprocessed dataset saved successfully: E:\HOPE\AI Course Tamil\translation_project\data\preprocessed_dataset.csv


In [25]:
df.head()

,id,input_text,target_language,human_translated_text,clean_text,detected_lang,moderation_status
0,1,அழுமூஞ்சியாக இருக்காதே.,en,Don’t be a cry-baby. / Don’t be a fusspot.,அழுமூஞ்சியாக இருக்காதே.,ta,approved
1,2,சிணுங்குவதை நிறுத்து.,en,Stop whining. / Stop whimpering.,சிணுங்குவதை நிறுத்து.,ta,approved
2,3,நீ மிகவும் குறும்புக்காரி.,en,You are very naughty.,நீ மிகவும் குறும்புக்காரி.,ta,approved
3,4,இந்த பையன் மிகவும் தேட்தடச் சேயவன்.,en,This boy is very mischievous; troublesome.,இந்த பையன் மிகவும் தேட்தடச் சேயவன்.,ta,approved
4,5,ஒழுங்கா இரு.,en,Behave yourself. / Keep quiet.,ஒழுங்கா இரு.,ta,approved


In [26]:
#Save Preprocessed Output

# Create folder if it doesn't exist
os.makedirs("../data", exist_ok=True)

# Save preprocessed file
df.to_csv("../data/preprocessed_dataset.csv", index=False, encoding="utf-8")
logger.info("Preprocessed dataset saved successfully.")

2025-11-03 15:06:40.008 | INFO     | __main__:<module>:8 - Preprocessed dataset saved successfully.
